# PPMI Dataset Exploration

Initial exploration of the PPMI Parkinson's disease dataset.

**Dataset**: PPMI Curated Data Cut (March 21, 2025)
- 15,316 patient visits
- 181 features
- Longitudinal data (up to 13 years)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

## 1. Load Data

In [ ]:
# Define data paths
data_dir = Path('../../PPMI_Curated_Data_Cut_Public_20250321')
data_file = data_dir / '20250310-Table 1.csv'
dict_file = data_dir / 'Data dictionary-Table 1.csv'
info_file = data_dir / 'Information-Table 1.csv'

# Load main dataset
df = pd.read_csv(data_file)
print(f"Dataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Load data dictionary
data_dict = pd.read_csv(dict_file)
print(f"Data dictionary shape: {data_dict.shape}")
data_dict.head()

## 2. Basic Data Overview

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Data types
print("Data types:")
print(df.dtypes.value_counts())
print("\nColumn data types:")
df.dtypes

In [ ]:
# Basic statistics
df.describe()

## 3. Cohort Distribution

In [ ]:
# Check cohort distribution
print("Cohort distribution:")
print(df['COHORT'].value_counts())
print(f"\nTotal visits: {len(df)}")
print(f"Unique patients: {df['PATNO'].nunique()}")

In [ ]:
# Visualize cohort distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count by cohort
df['COHORT'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Visit Counts by Cohort')
axes[0].set_xlabel('Cohort')
axes[0].set_ylabel('Number of Visits')
axes[0].tick_params(axis='x', rotation=45)

# Unique patients by cohort
df.groupby('COHORT')['PATNO'].nunique().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Unique Patients by Cohort')
axes[1].set_xlabel('Cohort')
axes[1].set_ylabel('Number of Patients')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. Visit Distribution

In [ ]:
# Visit event distribution
print("Visit event distribution:")
print(df['EVENT_ID'].value_counts().sort_index())

In [ ]:
# Visualize visits over time
fig, ax = plt.subplots(figsize=(14, 6))
visit_counts = df['EVENT_ID'].value_counts().sort_index()
visit_counts.plot(kind='bar', ax=ax, color='seagreen')
ax.set_title('Visit Distribution Across Timepoints')
ax.set_xlabel('Visit Event')
ax.set_ylabel('Number of Records')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 5. Missing Data Analysis

In [ ]:
# Calculate missing percentages
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print("Top 20 features with most missing data:")
print(missing_pct.head(20))

In [ ]:
# Visualize missing data
fig, ax = plt.subplots(figsize=(14, 8))
top_missing = missing_pct[missing_pct > 0].head(30)
top_missing.plot(kind='barh', ax=ax, color='crimson')
ax.set_title('Top 30 Features with Missing Data')
ax.set_xlabel('Missing Percentage (%)')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()

## 6. Feature Categories

Explore different categories of features in the dataset.

In [ ]:
# List all column names
print("All features:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:3d}. {col}")

In [ ]:
# Identify feature categories (based on naming patterns)
biomarker_cols = [col for col in df.columns if any(marker in col.lower() for marker in 
                  ['tau', 'abeta', 'synuclein', 'urate', 'neurofilament', 'plasma', 'csf', 'serum'])]
imaging_cols = [col for col in df.columns if any(img in col.lower() for img in 
                ['caudate', 'putamen', 'striatum', 'datscan'])]
updrs_cols = [col for col in df.columns if 'updrs' in col.lower()]
cognitive_cols = [col for col in df.columns if any(cog in col.lower() for cog in 
                  ['moca', 'hvlt', 'naming', 'clock', 'trail', 'letter', 'symbol', 'benton'])]
genetic_cols = [col for col in df.columns if 'apoe' in col.lower()]

print(f"Biomarker features: {len(biomarker_cols)}")
print(f"Imaging features: {len(imaging_cols)}")
print(f"UPDRS features: {len(updrs_cols)}")
print(f"Cognitive features: {len(cognitive_cols)}")
print(f"Genetic features: {len(genetic_cols)}")

## 7. Demographics

In [ ]:
# Age distribution
if 'age' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    df['age'].hist(bins=30, ax=axes[0], color='skyblue', edgecolor='black')
    axes[0].set_title('Age Distribution (All Visits)')
    axes[0].set_xlabel('Age')
    axes[0].set_ylabel('Frequency')
    
    df.boxplot(column='age', by='COHORT', ax=axes[1])
    axes[1].set_title('Age Distribution by Cohort')
    axes[1].set_xlabel('Cohort')
    axes[1].set_ylabel('Age')
    plt.suptitle('')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Sex distribution
if 'sex' in df.columns:
    print("Sex distribution:")
    print(df['sex'].value_counts())
    print("\nSex by cohort:")
    print(pd.crosstab(df['COHORT'], df['sex'], normalize='index') * 100)

## 8. Key Clinical Variables

In [ ]:
# UPDRS scores
if updrs_cols:
    print("UPDRS scores summary:")
    print(df[updrs_cols].describe())

In [ ]:
# MoCA scores (cognitive)
if 'moca_total' in df.columns:
    fig, ax = plt.subplots(figsize=(12, 6))
    df.boxplot(column='moca_total', by='COHORT', ax=ax)
    ax.set_title('MoCA Total Score by Cohort')
    ax.set_xlabel('Cohort')
    ax.set_ylabel('MoCA Score')
    plt.suptitle('')
    plt.tight_layout()
    plt.show()

## 9. New Features (March 2025)

Explore newly added NSD-ISS staging and plasma biomarkers.

In [ ]:
# NSD-ISS staging variables
nsd_cols = [col for col in df.columns if 'nsd' in col.lower()]
print(f"NSD-ISS features: {len(nsd_cols)}")
print(nsd_cols)

if nsd_cols:
    print("\nNSD-ISS summary:")
    print(df[nsd_cols].describe())

In [ ]:
# Plasma biomarkers
plasma_cols = [col for col in df.columns if 'plasma' in col.lower()]
print(f"Plasma biomarker features: {len(plasma_cols)}")
print(plasma_cols)

if plasma_cols:
    print("\nPlasma biomarker summary:")
    print(df[plasma_cols].describe())

## 10. Next Steps

Based on this initial exploration:
1. Investigate newly added NSD-ISS staging variables
2. Analyze plasma biomarkers (ptau217, bd_tau)
3. Examine temporal progression patterns
4. Feature selection and importance analysis
5. Build predictive models
6. Compare with previous findings